In [ ]:
"""            
5. For each contig:
        Do blast, check if the contig is part of valid yeast sequence
        
        Each read in part 2 becomes a node in a graph. Each line(edge) between nodes 
        represents the overlap value. The sequencing problem becomes the traveling
        salesmen problem, visiting every graph node with the largest sum of overlap
        values. You can try a greedy algorithm. Start with the largest overlap value.
        Follow the path with the largest overlap values, which produces one contig.
        If traversal is not possible any longer, you can start from another point in
        the graph to produce another contig.
        
        
6. List contigs generated in step 3, perform blast search of contigs, and show how 
    contigs are searched to be part of Yeast chromosome 1 sequence
    
    
7. Repeat steps 1 through 4 with reads mixed from two chromosomes, 1 and 2. 
    Determine how many contigs are valid chromosome 1 or 2 segments
"""

In [23]:
from Bio import Entrez, Seq, SeqIO
from Bio.Alphabet import IUPAC
from collections import defaultdict
from Bio.SeqUtils import GC

import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
import scipy.stats
import random

plt.style.use('ggplot')

%matplotlib inline

# Step 1: Input sequences

## Chromosome 1

In [24]:
sequence1 = SeqIO.read(open('sequence1.fasta'), 'fasta')
print(sequence1)
print("Length:", len(sequence1), '\n')
print(sequence1.seq[:230])

ID: gi|330443391|ref|NC_001133.9|
Name: gi|330443391|ref|NC_001133.9|
Description: gi|330443391|ref|NC_001133.9| Saccharomyces cerevisiae S288C chromosome I, complete sequence
Number of features: 0
Seq('CCACACCACACCCACACACCCACACACCACACCACACACCACACCACACCCACA...GGG', SingleLetterAlphabet())
Length: 230218 

CCACACCACACCCACACACCCACACACCACACCACACACCACACCACACCCACACACACACATCCTAACACTACCCTAACACAGCCCTAATCTAACCCTGGCCAACCTGTCTCTCAACTTACCCTCCATTACCCTGCCTCCACTCGTTACCCTGTCCCATTCAACCATACCACTCCGAACCACCATCCATCCCTCTACTTACTACCACTCACCCACCGTTACCCTCCAAT


In [25]:
records1 = SeqIO.parse('sequence1.fasta', 'fasta')
for rec in records1:
    sequence1 = rec.seq

## Chromosome 2

In [26]:
sequence2 = SeqIO.read(open('sequence2.fasta'), 'fasta')
print(sequence2)
print("Length:", len(sequence2), '\n')
print(sequence2.seq[:230])

ID: gi|330443482|ref|NC_001134.8|
Name: gi|330443482|ref|NC_001134.8|
Description: gi|330443482|ref|NC_001134.8| Saccharomyces cerevisiae S288C chromosome II, complete sequence
Number of features: 0
Seq('AAATAGCCCTCATGTACGTCTCCTCCAAGCCCTGTTGTCTCTTACCCGGATGTT...TGT', SingleLetterAlphabet())
Length: 813184 

AAATAGCCCTCATGTACGTCTCCTCCAAGCCCTGTTGTCTCTTACCCGGATGTTCAACCAAAAGCTACTTACTACCTTTATTTTATGTTTACTTTTTATAGGTTGTCTTTTTATCCCACTTCTTCGCACTTGTCTCTCGCTACTGCCGTGCAACAAACACTAAATCAAAACAATGAAATACTACTACATCAAAACGCATTTTCCCTAGAAAAAAAATTTTCTTACAATAT


In [27]:
records2 = SeqIO.parse('sequence2.fasta', 'fasta')
for rec in records2:
    sequence2 = rec.seq

# Step 2: Replace N's with any base

## Chromosome 1

In [28]:
Sequence1 = Seq.Seq(str(sequence1), IUPAC.unambiguous_dna)
print(sequence1[:230])

CCACACCACACCCACACACCCACACACCACACCACACACCACACCACACCCACACACACACATCCTAACACTACCCTAACACAGCCCTAATCTAACCCTGGCCAACCTGTCTCTCAACTTACCCTCCATTACCCTGCCTCCACTCGTTACCCTGTCCCATTCAACCATACCACTCCGAACCACCATCCATCCCTCTACTTACTACCACTCACCCACCGTTACCCTCCAAT


In [29]:
Sequence1

Seq('CCACACCACACCCACACACCCACACACCACACCACACACCACACCACACCCACA...GGG', IUPACUnambiguousDNA())

In [30]:
Sequence1.find('N')

-1

In [31]:
Sequence1.count("N")

0

## Chromosome 2

In [32]:
Sequence2 = Seq.Seq(str(sequence2), IUPAC.unambiguous_dna)
print(sequence2[:230])

AAATAGCCCTCATGTACGTCTCCTCCAAGCCCTGTTGTCTCTTACCCGGATGTTCAACCAAAAGCTACTTACTACCTTTATTTTATGTTTACTTTTTATAGGTTGTCTTTTTATCCCACTTCTTCGCACTTGTCTCTCGCTACTGCCGTGCAACAAACACTAAATCAAAACAATGAAATACTACTACATCAAAACGCATTTTCCCTAGAAAAAAAATTTTCTTACAATAT


In [33]:
Sequence2

Seq('AAATAGCCCTCATGTACGTCTCCTCCAAGCCCTGTTGTCTCTTACCCGGATGTT...TGT', IUPACUnambiguousDNA())

In [34]:
Sequence2.find('N')

-1

In [35]:
Sequence2.count("N")

0

# Step 3: Generate Reads

## Chromosome 1

In [36]:
length1 = int(5*230218/400)
position1 = np.zeros(length1)

ignore_start_end1 = 230218 - 400
base_pair = 400

start = 0
end = 0

reads1 = ["" for i in range(length1)]

seq1 = str(Sequence1)

In [37]:
for x in range(length1):
    position1[x] = random.randint(0, ignore_start_end1)
position1

array([112439., 170947.,  86460., ..., 138426.,  20037., 122340.])

In [38]:
for key in range(length1):
    start = int(position1[key])
    end = start + base_pair
    reads1[key] = seq1[start:end]

In [42]:
for x in range(10):
    print(x, ":", reads1[x], '\n')

0 : TTCTTGTGTTGTAAGAGCGCCTGGTTGGCTTCGTCCTTGCCATTAGCAAACAGCTCTATCTTGGGAGTCAATGGTGTAGAAACTGCACTGTTAGTTGCCGTGCCTGAGGGAGTGCTATCTCCCGTAGGTTGTTGTTTCTTTGCTGTCTTGCTGTCGGTGAGAGTGTCGGCCATTTCCATCAAGGCTTGCTTTGTGCAGTCTACCAGTGATTTGGAGGCTTCTGCAATGTGAGATTGCTGGAAATCGGTGGGCTGCAAAGCTGGTGGTGCAGCCTGAGGTCCGGGGCCAAATGGGCCCGGGACCTGCTGCTGTGCCTGTTGCTGCGCTTGTTGCTGAGCTTGCTGAGCCTGTTGTGTAGCCAAATACTTCTTCATAGCGTTTTGTCTAGCATAAATGTTGG 

1 : TGGTGTACCGACGTATGTTGTGGCAAATTGAATACTAGTTTCCAGAGATTTGGCTAACCCAAAATCACCTAACTTTACCACAACTTGACTATAGTCCATAGGGCTCCCCCTTTTCCCTGAATTCACTCTATGGTCTCTGTAATAATTACTATTCACTTCCTCGTGACCGTCTACTTGTTCATTAATATTGTAATCGCTATCATCATAGCTTAAGAATATATTTCCTGGTTTCAGATCACGATGGATAACGATGTTTTTGCCTTTTACCGGTGGTTTCATCCGGTCATATATTGTGGTCAAAGTTGGCAATTCAACACCATAATGACATTTATAGAGCGCAGTCAATAATTGGGCCAGGATACCCCACACAATTTTTTCTGGTATATATTTATGCTCCTGT 

2 : TTATTGAAGTGTCAAATGGAAGTGACCGAGCTTTGTACGCATCCCTTTCCAATAAAATCTGTCTCAACTTGGCATTATAAATTTTTATAGCATTTTGAGCAATGCAAGACAGCGAATTTAAAATCTTCCATTGTAGAGAATTATTGTATATGCACGATTTATAGTGATCTAGGGGTACCTCC

## Chromosome 2

In [43]:
length2 = int(5*813184/400)
position2 = np.zeros(length2)

ignore_start_end2 = 813184 - 400
base_pair = 400

start = 0
end = 0

reads2 = ["" for i in range(length2)]

seq2 = str(Sequence2)

In [44]:
for x in range(length2):
    position2[x] = random.randint(0, ignore_start_end2)
position2

array([175307., 364507., 204250., ...,  18489., 183942., 507987.])

In [45]:
for key in range(length2):
    start = int(position2[key])
    end = start + base_pair
    reads2[key] = seq2[start:end]

In [46]:
for x in range(10):
    print(x, ":", reads2[x], '\n')

0 : GCATTAAGCTGCTCATTTATTTCATCTTCTCCTTGTTCTATCGCCGATTCGCCATTATTTTTAAGCTCTTCGCCTTCTCGATCTTCATCGTTTTCTGGATGAGATCTTACATGAGAATCGACAACAAATGTGGCCAATCTTTCGTCTGCCTCTTCATCAACAAGGTCTCTGACAACACATAAAATATCAAATCTAGACAGAATAGGCTCGGTCAAACTAACATTCTGAGCTAAAGGCAAGGTTGAATTATATCTACCACCATTAGGATTTGCCGCAGCAATAATTGAGCAGCGCGCTTGTAATGTAGTAACAATACCGGCCTTGGAAATGGAAATACTTTGCTGTTCCATAGCCTCATGAATAGATGTACGATCCTGATCGTTCATCTTATCGAATTCAT 

1 : CAATTCGAAAGTTCTATTACTCGCAAATTCGAAAGCAGATGTTTTAAGAGACGAGGTGAAGTAAAAAAGAAGAAAAATGATTGCCATACTCCTCAATGCTTGCATGTAATATAATATAATATAATTACTGTTTCATTTTCTTTTATATTATACAATGGATATTCTACATAACCGTAACCTATATCGTTTTTAATGATTGCAAGGAAAAGTTGATTTTGTAGATAATCTTATAATGTAGTGACTTATTTAATTGTTTGATTAGAGGTAGTAATATTGCAATCAAACTGATCTAGTGAGTTTCCCGCTCCTTTTTAATTCCAAAGCTTTCTTGTAGGGAGGGTTCGTCGGGCTTTGAACAGGATCTAAGGACGATGAAGTGCCTGCAACCGAAGAAGGTAGG 

2 : TGAGCATTTGGCCCATAACAACTGCATTGGGATCTTTCCTGAAGGTGGGTCCCACGACAGAACAAACTTGTTGCCCCTGAAAGCAGGTGTGGCGATTATGGCTCTTGGTTGCATGGATAAGCATCCTGACGTCAATGTTAAGATTGTTCCCTGCGGTATGAATTATTTCCATCCACATAAGT

# Step 4: OLC Sequencing

## Chromosome 1

4. Do OLC sequencing

    (reads from one chromosome only)
    
    Write a program(s) to create the overlap graph from reads. An overlap(si, sj) is
    defined as the length of the longest matches between the suffix of si and the prefix
    of sj. In order to compute the overlaps, you need to perform n*(n-1) comparisons,
    where n denotes the number of reads
    
    The following is an example with 7 reads, and the overlap value is in the parentheses.
    
              read      overlap
        1. TACCTTG      2(3)  4(1)  7(1)
        2. TTGAT        3(3)
        3. GATATGG      4(2) 7(1)
        4. GGAG         3(1) 7(1)
        5. CTCTA        1(2)
        6. CTAGT        1(1) 2(1) 
        7. GCTCT        2(1) 5(4) 6(2)